# Indagine sulle Classi di Oggetti SDSS: STELLA, GALASSIA, QUASAR

## Introduzione

Lo **Sloan Digital Sky Survey (SDSS)** classifica gli oggetti celesti in diverse categorie in base al loro spettro. Le principali classi spettroscopiche sono:

- **STAR** — Stella: oggetti con spettro stellare, caratterizzato da righe di assorbimento tipiche delle atmosfere stellari.
- **GALAXY** — Galassia: sistemi di stelle, gas e polveri, con spettri che mostrano righe di assorbimento e/o emissione integrate su tutta la popolazione stellare.
- **QSO** — Quasar: nuclei galattici attivi estremamente luminosi e distanti, con spettri caratterizzati da larghe righe di emissione.

SDSS **non** ha una classe `PLANET` nel catalogo spettroscopico `SpecObj`, poiché gli esopianeti vengono scoperti con metodi indiretti (transito, velocità radiale, microlensing) e non con spettroscopia diretta.

In questo notebook:
1. Eseguiamo query per ottenere 100.000 oggetti delle classi STAR, GALAXY, QSO
2. Esploriamo la distribuzione delle classi
3. Verifichiamo che la classe PLANET non esista
4. Identifichiamo tutte le classi disponibili in SpecObj

In [ ]:
from astroquery.sdss import SDSS
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)

## 1. Query principale: 100.000 oggetti (STAR, GALAXY, QSO)

Eseguiamo una query per ottenere 100.000 oggetti con redshift tra 0.01 e 1, appartenenti alle classi STAR, GALAXY o QSO. Joiniamo le tabelle `PhotoObjAll` (dati fotometrici) e `SpecObj` (classificazione spettroscopica e redshift).

In [ ]:
query = """
SELECT TOP 100000
    p.objid, 
    p.ra, 
    p.dec, 
    s.z AS redshift, 
    s.class,
    p.u, 
    p.g, 
    p.r, 
    p.i, 
    p.z
FROM
    PhotoObjAll AS p
JOIN
    SpecObj AS s ON s.bestobjid = p.objid
WHERE
    s.zWarning = 0
AND
    s.z BETWEEN 0.01 AND 1
AND
    s.class IN ('STAR', 'GALAXY', 'QSO')
ORDER BY
    s.z DESC
"""

result = SDSS.query_sql(query)
df = result.to_pandas()
df.to_csv("sdss_star_galaxy_qso.csv", index=False)
df.head()

## 2. Esplorazione delle classi

Esaminiamo la distribuzione delle classi nel dataset.

In [ ]:
class_counts = df['class'].value_counts()
print("Distribuzione delle classi:")
print(class_counts)

class_counts.plot(kind='bar', color=['gold', 'steelblue', 'coral'], edgecolor='black')
plt.title('Distribuzione delle classi SDSS (100.000 oggetti)')
plt.xlabel('Classe')
plt.ylabel('Conteggio')
plt.grid(alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.show()

## 3. Verifica dell'esistenza della classe PLANET

SDSS non classifica pianeti nel catalogo spettroscopico. Verifichiamo che non esista alcun oggetto con `class = 'PLANET'`.

In [ ]:
df_planets = df[df['class'] == 'PLANET']
print(f"Oggetti con classe 'PLANET' nel risultato: {len(df_planets)}")

count_query = """SELECT COUNT(*) FROM SpecObj WHERE class = 'PLANET'"""
count_result = SDSS.query_sql(count_query)
print("Conteggio totale in SpecObj per class='PLANET':")
print(count_result)

## 4. Classi disponibili in SpecObj

Elenchiamo tutte le classi distinte presenti nel catalogo `SpecObj` di SDSS.

In [ ]:
class_query = """SELECT DISTINCT class FROM SpecObj"""
result_classes = SDSS.query_sql(class_query)
print("Classi disponibili in SpecObj:")
print(result_classes)

## Conclusioni

Come previsto, la classe `PLANET` **non esiste** in SDSS SpecObj. Le uniche classi disponibili sono:
- **STAR** (stelle)
- **GALAXY** (galassie)
- **QSO** (quasar)

Gli esopianeti vengono identificati tramite altre tecniche osservative (transiti, velocità radiali, imaging diretto, microlensing) e catalogati in database specializzati come il **NASA Exoplanet Archive** o il **Exoplanet.eu**.